# 0102 Update - Speech Emotion Data Processing v2

Notebook này là bản cập nhật cho phần **01&02 Data and Data Processing**.

Mục tiêu chính:

- xử lý 4 dataset: `RAVDESS`, `CREMA-D`, `TESS`, `SAVEE`;
- chuẩn hóa metadata cho 6 nhãn chung: `neutral`, `happy`, `sad`, `angry`, `fear`, `disgust`;
- tạo nhiều phiên bản audio để tránh mất tín hiệu cảm xúc:
  - `audio_16k_raw`: chỉ resample mono 16 kHz, không trim/normalize mạnh;
  - `audio_16k_clean_full`: trim nhẹ + RMS normalization, giữ độ dài tự nhiên;
  - `audio_16k`: model-ready 4s crop/pad để tương thích notebook 04/05/06;
- lưu thêm metadata về duration, RMS, peak, silence/pause ratio để phục vụ EDA và feedback;
- trích xuất baseline feature vector rõ ràng hơn;
- tạo package zip nhẹ để tải report/metadata/features về, không zip audio nặng mặc định.

Điểm mới so với bản cũ:

```text
Không chỉ tạo audio_16k model-ready.
Giữ thêm raw-resampled và clean-full để sau này phân tích pause/energy thật hơn.
Không dùng peak normalization cứng làm nguồn duy nhất.
Thêm RMS/loudness-style normalization và metadata đo năng lượng/silence.
```

## 1. Vì Sao Preprocessing Có Thể Ảnh Hưởng Accuracy?

Speech emotion recognition không chỉ phụ thuộc model. Cách xử lý audio có thể làm mất hoặc giữ lại các cue cảm xúc:

- `energy/loudness`: angry/happy thường mạnh hơn, sad/neutral thường thấp hơn;
- `pause/silence`: ngập ngừng, run, thiếu tự tin có thể thể hiện qua khoảng nghỉ;
- `pitch contour`: cao độ và biến thiên cao độ thể hiện trạng thái cảm xúc;
- `duration/rhythm`: tốc độ nói và nhịp nói cũng là tín hiệu.

Vì vậy notebook này tạo nhiều audio version:

| Version | Cách xử lý | Dùng cho |
|---|---|---|
| `audio_16k_raw` | mono + resample 16 kHz + remove DC offset | đo pause/energy gần gốc, EDA, feedback |
| `audio_16k_clean_full` | trim nhẹ + RMS normalization + peak guard | trích feature sạch nhưng còn độ dài tự nhiên |
| `audio_16k` | clean + crop/pad 4s | tương thích notebook train hiện tại |

Lưu ý: model 06D/06F vẫn có thể tự xử lý thêm trong notebook model. Nhưng dữ liệu đầu vào tốt hơn giúp model không bị giới hạn bởi preprocessing quá mạnh.

In [ ]:
from pathlib import Path
import os
import re
import json
import random
import shutil
import zipfile
import warnings

import numpy as np
import pandas as pd
import librosa
import librosa.display
import soundfile as sf
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = int(os.getenv("SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)

print("SEED:", SEED)

## 2. Chuẩn Bị Hoặc Tải Dataset

Notebook ưu tiên dùng dataset đã attach sẵn trên Kaggle, vì đây là cách ổn định nhất khi chạy notebook hoặc Save Version.

Thứ tự tìm dataset:

1. Kaggle Input đã attach sẵn, ví dụ `/kaggle/input/datasets/quanghuy225/tripe-dataset` hoặc `/kaggle/input/tripe-dataset`.
2. Nếu chưa attach dataset và `AUTO_DOWNLOAD_DATASETS=1`, notebook thử tải Kaggle dataset bằng slug `quanghuy225/tripe-dataset`.
3. Nếu không dùng Kaggle, notebook tìm trong folder local `datasets`.

Gợi ý trên Kaggle:

- Nếu bạn đã Add Data dataset `quanghuy225/tripe-dataset`, không cần tải lại.
- Nếu muốn notebook tự tải, bật Internet và giữ `AUTO_DOWNLOAD_DATASETS=1`.
- Nếu Kaggle yêu cầu token/API, cách chắc nhất vẫn là Add Data ở panel bên phải.

In [ ]:
def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None

PROJECT_ROOT = Path("/kaggle/working/Speech_Project") if Path("/kaggle/working").exists() else Path.cwd().resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET_SLUG = os.getenv("KAGGLE_DATASET_SLUG", "quanghuy225/tripe-dataset")
AUTO_DOWNLOAD_DATASETS = os.getenv("AUTO_DOWNLOAD_DATASETS", "1") == "1"
DOWNLOAD_DATA_ROOT = PROJECT_ROOT / "datasets_downloaded"

KNOWN_DATASET_KEYS = ["ravdess", "crema", "crema-d", "tess", "savee"]


def looks_like_dataset_root(path):
    path = Path(path)
    if not path.exists() or not path.is_dir():
        return False
    names = [p.name.lower() for p in path.iterdir() if p.is_dir()]
    matched = 0
    for key in KNOWN_DATASET_KEYS:
        if any(key in name for name in names):
            matched += 1
    return matched >= 2


def discover_dataset_root(base_paths, max_depth=2):
    checked = []
    for base in base_paths:
        base = Path(base)
        if not base.exists():
            continue
        queue = [(base, 0)]
        while queue:
            current, depth = queue.pop(0)
            checked.append(str(current))
            if looks_like_dataset_root(current):
                return current, checked
            if depth < max_depth and current.is_dir():
                try:
                    for child in current.iterdir():
                        if child.is_dir():
                            queue.append((child, depth + 1))
                except PermissionError:
                    pass
    return None, checked


def try_download_kaggle_dataset(slug, download_dir):
    download_dir = Path(download_dir)
    download_dir.mkdir(parents=True, exist_ok=True)
    print(f"Trying to download Kaggle dataset: {slug}")

    try:
        import kagglehub
        downloaded_path = Path(kagglehub.dataset_download(slug))
        print("Downloaded by kagglehub:", downloaded_path)
        return downloaded_path
    except Exception as exc:
        print("kagglehub download failed:", repr(exc))

    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files(slug, path=str(download_dir), unzip=True, quiet=False)
        print("Downloaded by Kaggle API:", download_dir)
        return download_dir
    except Exception as exc:
        print("Kaggle API download failed:", repr(exc))

    return None

DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/quanghuy225/tripe-dataset"),
    Path("/kaggle/input/datasets/tripe-dataset"),
    Path("/kaggle/input/tripe-dataset"),
    Path("/kaggle/input"),
    PROJECT_ROOT / "01&02_Data_and_DataProcessing" / "datasets",
    PROJECT_ROOT / "datasets",
    Path.cwd().resolve() / "01&02_Data_and_DataProcessing" / "datasets",
    Path.cwd().resolve() / "datasets",
    Path.cwd().resolve().parent / "datasets",
    DOWNLOAD_DATA_ROOT,
]

DATA_ROOT, checked_roots = discover_dataset_root(DATA_ROOT_CANDIDATES, max_depth=2)
DATA_SOURCE_MODE = "attached_or_local"

if DATA_ROOT is None and AUTO_DOWNLOAD_DATASETS:
    downloaded_root = try_download_kaggle_dataset(KAGGLE_DATASET_SLUG, DOWNLOAD_DATA_ROOT)
    if downloaded_root is not None:
        DATA_ROOT, checked_after_download = discover_dataset_root([downloaded_root, DOWNLOAD_DATA_ROOT], max_depth=3)
        checked_roots.extend(checked_after_download)
        DATA_SOURCE_MODE = "downloaded"

if DATA_ROOT is None:
    print("Checked dataset roots:")
    for item in checked_roots[:40]:
        print(" -", item)
    raise FileNotFoundError(
        "Cannot find RAVDESS/CREMA-D/TESS/SAVEE folders. "
        "On Kaggle, attach dataset quanghuy225/tripe-dataset or enable Internet/API download."
    )

OUTPUT_DIR = PROJECT_ROOT / "ser_processed_update"
RAW_AUDIO_DIR = OUTPUT_DIR / "audio_16k_raw"
CLEAN_AUDIO_DIR = OUTPUT_DIR / "audio_16k_clean_full"
MODEL_AUDIO_DIR = OUTPUT_DIR / "audio_16k"
REPORT_DIR = OUTPUT_DIR / "reports"
FEATURE_DIR = OUTPUT_DIR / "features"
FIGURE_DIR = OUTPUT_DIR / "figures"

for d in [OUTPUT_DIR, RAW_AUDIO_DIR, CLEAN_AUDIO_DIR, MODEL_AUDIO_DIR, REPORT_DIR, FEATURE_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TARGET_SR = int(os.getenv("TARGET_SR", "16000"))
TARGET_DURATION = float(os.getenv("TARGET_DURATION", "4.0"))
TARGET_LENGTH = int(TARGET_SR * TARGET_DURATION)

TRIM_TOP_DB = float(os.getenv("TRIM_TOP_DB", "30"))
MIN_AUDIO_SEC_AFTER_TRIM = float(os.getenv("MIN_AUDIO_SEC_AFTER_TRIM", "0.35"))
TARGET_RMS_DBFS = float(os.getenv("TARGET_RMS_DBFS", "-23.0"))
TARGET_RMS = 10 ** (TARGET_RMS_DBFS / 20.0)
PEAK_GUARD = float(os.getenv("PEAK_GUARD", "0.99"))

CACHE_AUDIO = os.getenv("CACHE_AUDIO", "1") == "1"
SAVE_LOG_MEL_SAMPLE = os.getenv("SAVE_LOG_MEL_SAMPLE", "1") == "1"
QUICK_RUN = os.getenv("QUICK_RUN", "0") == "1"
QUICK_RUN_PER_DATASET = int(os.getenv("QUICK_RUN_PER_DATASET", "120"))

COMMON_EMOTIONS = ["neutral", "happy", "sad", "angry", "fear", "disgust"]
LABEL_TO_ID = {label: i for i, label in enumerate(COMMON_EMOTIONS)}

print("KAGGLE_DATASET_SLUG:", KAGGLE_DATASET_SLUG)
print("AUTO_DOWNLOAD_DATASETS:", AUTO_DOWNLOAD_DATASETS)
print("DATA_SOURCE_MODE:", DATA_SOURCE_MODE)
print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("TARGET_SR:", TARGET_SR, "TARGET_DURATION:", TARGET_DURATION)
print("CACHE_AUDIO:", CACHE_AUDIO, "QUICK_RUN:", QUICK_RUN)

## 3. Tìm File Audio Và Chuẩn Hóa Metadata

Notebook tự tìm folder dataset trên Kaggle/local. Dataset có thể nằm dưới các tên khác nhau:

- `RAVDESS`
- `CREMA-D` hoặc `AudioWAV`
- `TESS`
- `SAVEE` hoặc `ALL`

Cơ chế dedup chỉ giữ file audio unique theo tên file để tránh trường hợp giải nén lồng folder làm nhân đôi dữ liệu.

In [ ]:
AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3"}

def find_dataset_dirs(aliases, search_roots=None):
    search_roots = search_roots or [DATA_ROOT, Path("/kaggle/input"), PROJECT_ROOT, Path.cwd().resolve(), Path.cwd().resolve().parent]
    found = []
    seen = set()
    alias_set = {a.lower() for a in aliases}
    for root in search_roots:
        if not root.exists():
            continue
        candidates = [root / alias for alias in aliases]
        for path in root.rglob("*"):
            if path.is_dir() and path.name.lower() in alias_set:
                candidates.append(path)
        for candidate in candidates:
            if candidate.exists() and candidate.is_dir():
                resolved = candidate.resolve()
                key = str(resolved).lower()
                if key not in seen:
                    seen.add(key)
                    found.append(resolved)
    pruned = []
    for candidate in sorted(found, key=lambda p: len(p.parts)):
        if not any(parent in candidate.parents or candidate == parent for parent in pruned):
            pruned.append(candidate)
    return pruned

def audio_files(root):
    if not Path(root).exists():
        return []
    files = sorted((p for p in Path(root).rglob("*") if p.suffix.lower() in AUDIO_EXTENSIONS),
                   key=lambda p: (len(p.relative_to(root).parts), str(p).lower()))
    unique = {}
    duplicates = []
    for path in files:
        key = path.name.lower()
        if key in unique:
            duplicates.append(path)
            continue
        unique[key] = path
    if duplicates:
        print(f"[Dedup] {Path(root).name}: ignored {len(duplicates):,} duplicate nested files; kept {len(unique):,}.")
    return list(unique.values())

RAVDESS_DIRS = find_dataset_dirs(["RAVDESS", "ravdess", "audio_speech_actors_01-24"])
CREMAD_DIRS = find_dataset_dirs(["CREMA-D", "CREMAD", "crema-d", "AudioWAV"])
TESS_DIRS = find_dataset_dirs(["TESS", "tess", "TESS Toronto emotional speech set data"])
SAVEE_DIRS = find_dataset_dirs(["SAVEE", "Savee", "savee", "ALL"])

print("RAVDESS_DIRS:", RAVDESS_DIRS)
print("CREMAD_DIRS:", CREMAD_DIRS)
print("TESS_DIRS:", TESS_DIRS)
print("SAVEE_DIRS:", SAVEE_DIRS)

In [ ]:
RAVDESS_EMOTION = {
    "01": "neutral", "02": "calm", "03": "happy", "04": "sad",
    "05": "angry", "06": "fear", "07": "disgust", "08": "surprise",
}
CREMAD_EMOTION = {
    "NEU": "neutral", "HAP": "happy", "SAD": "sad",
    "ANG": "angry", "FEA": "fear", "DIS": "disgust",
}
TESS_EMOTION = {
    "neutral": "neutral", "happy": "happy", "sad": "sad",
    "angry": "angry", "fear": "fear", "fearful": "fear",
    "disgust": "disgust", "disgusted": "disgust",
    "pleasant_surprise": "surprise", "surprise": "surprise",
}
SAVEE_EMOTION = {
    "a": "angry", "d": "disgust", "f": "fear", "h": "happy",
    "n": "neutral", "sa": "sad", "su": "surprise",
}

def parse_ravdess(path):
    parts = path.stem.split("-")
    if len(parts) < 7:
        return None
    emotion = RAVDESS_EMOTION.get(parts[2])
    actor = parts[6]
    return {
        "dataset": "RAVDESS",
        "speaker_id": f"ravdess_{actor}",
        "emotion": emotion,
        "original_emotion": parts[2],
        "gender": "male" if int(actor) % 2 else "female",
    }

def parse_cremad(path):
    parts = path.stem.split("_")
    if len(parts) < 3:
        return None
    return {
        "dataset": "CREMA-D",
        "speaker_id": f"cremad_{parts[0]}",
        "emotion": CREMAD_EMOTION.get(parts[2].upper()),
        "original_emotion": parts[2],
        "gender": "unknown",
    }

def parse_tess(path):
    stem = path.stem.lower().replace("ps", "pleasant_surprise")
    candidates = [k for k in TESS_EMOTION if stem.endswith("_" + k) or k in path.parent.name.lower()]
    if not candidates:
        return None
    original = max(candidates, key=len)
    actor_match = re.match(r"(oaf|yaf)", stem)
    actor = actor_match.group(1) if actor_match else path.parent.name.lower()
    return {
        "dataset": "TESS",
        "speaker_id": f"tess_{actor}",
        "emotion": TESS_EMOTION.get(original),
        "original_emotion": original,
        "gender": "female",
    }

def parse_savee(path):
    stem = path.stem.lower()
    code_match = re.match(r"([a-z]+)", stem)
    if not code_match:
        return None
    code = code_match.group(1)
    if code.startswith("sa"):
        emo_code = "sa"
    elif code.startswith("su"):
        emo_code = "su"
    else:
        emo_code = code[0]
    speaker = path.parent.name.lower()
    if speaker == "all":
        speaker = stem[:2] if len(stem) >= 2 else "unknown"
    return {
        "dataset": "SAVEE",
        "speaker_id": f"savee_{speaker}",
        "emotion": SAVEE_EMOTION.get(emo_code),
        "original_emotion": emo_code,
        "gender": "male",
    }

def build_metadata():
    rows = []
    parser_plan = [
        ("RAVDESS", RAVDESS_DIRS, parse_ravdess),
        ("CREMA-D", CREMAD_DIRS, parse_cremad),
        ("TESS", TESS_DIRS, parse_tess),
        ("SAVEE", SAVEE_DIRS, parse_savee),
    ]
    for dataset_name, dirs, parser in parser_plan:
        for root in dirs:
            for path in audio_files(root):
                parsed = parser(path)
                if not parsed:
                    continue
                rows.append({
                    "filepath": str(path),
                    "source_filename": path.name,
                    **parsed,
                })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No audio metadata found. Check dataset paths.")
    df = df[df["emotion"].isin(COMMON_EMOTIONS)].copy()
    df = df.drop_duplicates(["dataset", "source_filename"]).reset_index(drop=True)
    df.insert(0, "sample_id", [f"s{i:06d}" for i in range(len(df))])
    if QUICK_RUN:
        df = (
            df.groupby(["dataset", "emotion"], group_keys=False)
            .apply(lambda x: x.sample(min(len(x), max(2, QUICK_RUN_PER_DATASET // len(COMMON_EMOTIONS))), random_state=SEED))
            .reset_index(drop=True)
        )
        df["sample_id"] = [f"s{i:06d}" for i in range(len(df))]
    return df

metadata = build_metadata()
metadata["label_id"] = metadata["emotion"].map(LABEL_TO_ID).astype(int)
metadata["dataset_id"] = pd.Categorical(metadata["dataset"]).codes.astype(int)
metadata["speaker_global"] = metadata["dataset"].astype(str) + "::" + metadata["speaker_id"].astype(str)
metadata["speaker_id_int"] = pd.Categorical(metadata["speaker_global"]).codes.astype(int)

DOMAIN_NAMES = sorted(metadata["dataset"].unique().tolist())
DOMAIN_TO_ID = {name: i for i, name in enumerate(DOMAIN_NAMES)}
metadata["domain_id"] = metadata["dataset"].map(DOMAIN_TO_ID).astype(int)

display(metadata.head())
print("Samples:", len(metadata))
print("Datasets:", metadata["dataset"].value_counts().to_dict())
print("Emotion distribution:", metadata["emotion"].value_counts().to_dict())
print("Speakers:", metadata["speaker_global"].nunique())
print("Domains:", DOMAIN_NAMES)

## 4. Audio Inspection

Kiểm tra file đọc được, sample rate gốc, duration, peak, RMS.

Các file quá ngắn, silent hoặc không đọc được sẽ bị loại trước khi tạo split và cache.

In [ ]:
def inspect_audio(row):
    result = {
        "sample_id": row.sample_id,
        "readable": False,
        "error": "",
        "sample_rate": np.nan,
        "duration": np.nan,
        "channels": np.nan,
        "peak_raw_file": np.nan,
        "rms_raw_file": np.nan,
    }
    try:
        info = sf.info(row.filepath)
        y, _ = librosa.load(row.filepath, sr=None, mono=True, duration=30)
        result.update(
            readable=len(y) > 0,
            sample_rate=int(info.samplerate),
            duration=float(info.duration),
            channels=int(info.channels),
            peak_raw_file=float(np.max(np.abs(y))) if len(y) else 0.0,
            rms_raw_file=float(np.sqrt(np.mean(y ** 2))) if len(y) else 0.0,
        )
    except Exception as exc:
        result["error"] = str(exc)
    return result

validation = pd.DataFrame([inspect_audio(r) for r in tqdm(metadata.itertuples(index=False), total=len(metadata), desc="Inspect audio")])
validation.to_csv(REPORT_DIR / "audio_validation_report.csv", index=False)
metadata = metadata.merge(validation.drop(columns="error"), on="sample_id", how="left")

invalid = (~metadata["readable"].astype(bool)) | (metadata["duration"] < 0.25) | (metadata["peak_raw_file"] == 0)
print("Invalid/too-short/silent files:", int(invalid.sum()))
metadata = metadata.loc[~invalid].reset_index(drop=True)
display(metadata.describe(include="all").T)

In [ ]:
overview = pd.DataFrame({
    "samples": metadata.groupby("dataset").size(),
    "speakers": metadata.groupby("dataset")["speaker_id"].nunique(),
    "mean_duration_s": metadata.groupby("dataset")["duration"].mean(),
    "median_duration_s": metadata.groupby("dataset")["duration"].median(),
    "total_hours": metadata.groupby("dataset")["duration"].sum() / 3600,
}).round(3)
overview.to_csv(REPORT_DIR / "dataset_overview.csv")
display(overview)

emotion_dataset = pd.crosstab(metadata["emotion"], metadata["dataset"]).reindex(COMMON_EMOTIONS)
emotion_dataset.to_csv(REPORT_DIR / "emotion_by_dataset.csv")
display(emotion_dataset)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(data=metadata, x="dataset", ax=axes[0], palette="Set2")
axes[0].set_title("Samples by dataset")
sns.countplot(data=metadata, x="emotion", order=COMMON_EMOTIONS, ax=axes[1], palette="Set3")
axes[1].tick_params(axis="x", rotation=30)
axes[1].set_title("Samples by emotion")
sns.histplot(data=metadata, x="duration", hue="dataset", bins=50, ax=axes[2], element="step")
axes[2].set_title("Original duration distribution")
plt.tight_layout()
fig_path = FIGURE_DIR / "0102update_dataset_overview.png"
plt.savefig(fig_path, dpi=160)
plt.show()
print("Saved:", fig_path)

## 5. Speaker-Independent Split

Split theo speaker để hạn chế leakage:

```text
train speaker ∩ validation speaker = ∅
train speaker ∩ test speaker = ∅
validation speaker ∩ test speaker = ∅
```

Lưu ý: advanced notebooks 06D/06E/06F vẫn có thể tự tạo `combined_random`, `combined_strict_no_tess` và single-dataset protocols. Split trong metadata này chủ yếu phục vụ baseline/EDA và artifact thống nhất.

In [ ]:
def speaker_independent_split(df, test_size=0.20, val_size_from_remaining=0.125):
    groups = df["speaker_global"]
    outer = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)
    train_val_idx, test_idx = next(outer.split(df, groups=groups))
    train_val, test = df.iloc[train_val_idx].copy(), df.iloc[test_idx].copy()

    inner = GroupShuffleSplit(n_splits=1, test_size=val_size_from_remaining, random_state=SEED)
    train_idx, val_idx = next(inner.split(train_val, groups=train_val["speaker_global"]))
    train, val = train_val.iloc[train_idx].copy(), train_val.iloc[val_idx].copy()

    train["split"], val["split"], test["split"] = "train", "validation", "test"
    return pd.concat([train, val, test], ignore_index=True)

metadata = speaker_independent_split(metadata)

speaker_sets = {split: set(part["speaker_global"]) for split, part in metadata.groupby("split")}
assert speaker_sets["train"].isdisjoint(speaker_sets["validation"])
assert speaker_sets["train"].isdisjoint(speaker_sets["test"])
assert speaker_sets["validation"].isdisjoint(speaker_sets["test"])

split_summary = pd.crosstab(metadata["split"], metadata["emotion"]).reindex(["train", "validation", "test"])
split_summary.to_csv(REPORT_DIR / "speaker_independent_split_emotion.csv")
display(split_summary)
display(metadata.groupby("split").agg(samples=("sample_id", "count"), speakers=("speaker_global", "nunique")))

## 6. Audio Preprocessing v2

Ba phiên bản audio được tạo từ cùng một file gốc:

### `audio_16k_raw`

```text
load mono -> resample 16 kHz -> remove DC offset
```

Giữ silence và energy gần gốc để đo pause/energy cho feedback.

### `audio_16k_clean_full`

```text
raw -> trim silence nhẹ -> RMS normalize -> peak guard
```

Giữ độ dài tự nhiên sau trim, tốt cho feature extraction.

### `audio_16k`

```text
clean_full -> center crop hoặc pad thành 4s
```

Tương thích với model hiện tại, đặc biệt notebook 04/05/06 dùng `audio_16k/{sample_id}.wav`.

Notebook không dùng noise reduction mạnh vì có thể làm mất thông tin cảm xúc.

In [ ]:
def remove_dc(y):
    y = np.asarray(y, dtype=np.float32)
    if len(y):
        y = y - float(np.mean(y))
    return y.astype(np.float32)

def load_raw_16k(filepath):
    y, _ = librosa.load(filepath, sr=TARGET_SR, mono=True)
    return remove_dc(y)

def trim_silence(y):
    if len(y) == 0 or float(np.max(np.abs(y)) + 1e-8) < 1e-6:
        return y.astype(np.float32), np.array([[0, len(y)]], dtype=int)
    try:
        trimmed, intervals = librosa.effects.trim(y, top_db=TRIM_TOP_DB)
        if len(trimmed) >= int(MIN_AUDIO_SEC_AFTER_TRIM * TARGET_SR):
            return trimmed.astype(np.float32), intervals
    except Exception:
        pass
    return y.astype(np.float32), np.array([[0, len(y)]], dtype=int)

def rms_normalize(y):
    y = np.asarray(y, dtype=np.float32)
    if len(y) == 0:
        return y
    rms = float(np.sqrt(np.mean(y ** 2) + 1e-12))
    if rms > 1e-6:
        y = y * (TARGET_RMS / rms)
    peak = float(np.max(np.abs(y)) + 1e-8)
    if peak > PEAK_GUARD:
        y = y / peak * PEAK_GUARD
    return y.astype(np.float32)

def crop_or_pad(y, target_length=TARGET_LENGTH, mode="center"):
    y = np.asarray(y, dtype=np.float32)
    if len(y) < target_length:
        left = (target_length - len(y)) // 2
        right = target_length - len(y) - left
        return np.pad(y, (left, right)).astype(np.float32)
    if len(y) == target_length:
        return y.astype(np.float32)
    start = 0 if mode == "start" else (len(y) - target_length) // 2
    return y[start:start + target_length].astype(np.float32)

def audio_stats(y, prefix):
    y = np.asarray(y, dtype=np.float32)
    if len(y) == 0:
        return {
            f"{prefix}_duration": 0.0,
            f"{prefix}_peak": 0.0,
            f"{prefix}_rms": 0.0,
            f"{prefix}_mean_abs": 0.0,
        }
    return {
        f"{prefix}_duration": float(len(y) / TARGET_SR),
        f"{prefix}_peak": float(np.max(np.abs(y))),
        f"{prefix}_rms": float(np.sqrt(np.mean(y ** 2))),
        f"{prefix}_mean_abs": float(np.mean(np.abs(y))),
    }

def pause_stats(y):
    if len(y) == 0:
        return {"pause_ratio_raw": 1.0, "speech_ratio_raw": 0.0}
    intervals = librosa.effects.split(y, top_db=TRIM_TOP_DB)
    speech_samples = int(sum(end - start for start, end in intervals))
    speech_ratio = speech_samples / max(1, len(y))
    return {"pause_ratio_raw": float(1 - speech_ratio), "speech_ratio_raw": float(speech_ratio)}

def preprocess_versions(filepath):
    raw = load_raw_16k(filepath)
    trimmed, intervals = trim_silence(raw)
    clean = rms_normalize(trimmed)
    model = crop_or_pad(clean, TARGET_LENGTH, mode="center")
    stats = {}
    stats.update(audio_stats(raw, "raw16k"))
    stats.update(audio_stats(clean, "clean"))
    stats.update(audio_stats(model, "model4s"))
    stats.update(pause_stats(raw))
    stats["trim_removed_sec"] = max(0.0, len(raw) / TARGET_SR - len(trimmed) / TARGET_SR)
    stats["trim_intervals"] = int(len(intervals)) if intervals is not None else 0
    return raw, clean, model, stats

example = metadata.iloc[0]
raw_ex, clean_ex, model_ex, stats_ex = preprocess_versions(example.filepath)
print(example[["dataset", "emotion", "filepath"]].to_dict())
print(stats_ex)
print("raw/clean/model shapes:", raw_ex.shape, clean_ex.shape, model_ex.shape)

## 7. Feature Extraction For Baseline And Reports

Feature vector ở 0102 update phục vụ baseline/EDA. Các model 06D/06F vẫn tự trích feature sequence riêng trong notebook model.

Nhóm feature:

- MFCC, delta, delta-delta;
- RMS/energy;
- ZCR;
- spectral centroid, bandwidth, rolloff, contrast, flatness;
- chroma STFT;
- pitch/F0, voiced ratio;
- pause ratio;
- jitter/shimmer xấp xỉ;
- harmonic/percussive ratio.

Các feature được thống kê bằng mean/std/min/max/median/IQR để tạo vector cho classical ML.

In [ ]:
N_FFT = int(os.getenv("N_FFT", "400"))
WIN_LENGTH = int(os.getenv("WIN_LENGTH", "400"))
HOP_LENGTH = int(os.getenv("HOP_LENGTH", "160"))
N_MFCC = int(os.getenv("N_MFCC", "40"))
N_MELS = int(os.getenv("N_MELS", "96"))
MAX_FRAMES = int(1 + TARGET_LENGTH // HOP_LENGTH)

def safe_stats_1d(v):
    v = np.asarray(v, dtype=np.float32)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return np.zeros(6, dtype=np.float32)
    q25, q75 = np.percentile(v, [25, 75])
    return np.array([v.mean(), v.std(), v.min(), v.max(), np.median(v), q75 - q25], dtype=np.float32)

def summarize_matrix(mat):
    mat = np.asarray(mat, dtype=np.float32)
    q25 = np.percentile(mat, 25, axis=1)
    q75 = np.percentile(mat, 75, axis=1)
    return np.concatenate([
        mat.mean(axis=1), mat.std(axis=1), mat.min(axis=1), mat.max(axis=1),
        np.median(mat, axis=1), q75 - q25,
    ]).astype(np.float32)

def extract_pitch_and_voice_quality(y):
    extras = []
    try:
        f0, voiced_flag, _ = librosa.pyin(
            y, fmin=librosa.note_to_hz("C2"), fmax=librosa.note_to_hz("C7"),
            sr=TARGET_SR, frame_length=max(1024, N_FFT * 2), hop_length=HOP_LENGTH
        )
        f0_clean = f0[np.isfinite(f0)] if f0 is not None else np.array([])
        voiced_ratio = float(np.mean(voiced_flag)) if voiced_flag is not None else 0.0
    except Exception:
        f0_clean = np.array([])
        voiced_ratio = 0.0
    extras.append(safe_stats_1d(f0_clean))
    extras.append(np.array([voiced_ratio], dtype=np.float32))

    if f0_clean.size > 2:
        jitter = np.mean(np.abs(np.diff(f0_clean)) / (np.abs(f0_clean[:-1]) + 1e-6))
    else:
        jitter = 0.0

    rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)[0]
    rms_active = rms[rms > np.percentile(rms, 25)] if rms.size else np.array([])
    if rms_active.size > 2:
        shimmer = np.mean(np.abs(np.diff(rms_active)) / (np.abs(rms_active[:-1]) + 1e-6))
    else:
        shimmer = 0.0

    try:
        y_harm, y_perc = librosa.effects.hpss(y)
        harm_energy = np.mean(y_harm ** 2) + 1e-12
        perc_energy = np.mean(y_perc ** 2) + 1e-12
        hnr_like = 10.0 * np.log10(harm_energy / perc_energy)
        hp_ratio = harm_energy / (harm_energy + perc_energy)
    except Exception:
        hnr_like, hp_ratio = 0.0, 0.0
    extras.append(np.array([jitter, shimmer, hnr_like, hp_ratio], dtype=np.float32))
    return np.concatenate(extras).astype(np.float32)

def extract_feature_vector(y):
    mfcc = librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH)
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP_LENGTH)
    centroid = librosa.feature.spectral_centroid(y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    contrast = librosa.feature.spectral_contrast(y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)
    flatness = librosa.feature.spectral_flatness(y=y, n_fft=N_FFT, hop_length=HOP_LENGTH)
    chroma = librosa.feature.chroma_stft(y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH)

    matrix_features = [mfcc, delta, delta2, rms, zcr, centroid, bandwidth, rolloff, contrast, flatness, chroma]
    vector = np.concatenate([summarize_matrix(x) for x in matrix_features] + [extract_pitch_and_voice_quality(y)]).astype(np.float32)
    return vector

def extract_logmel_image(y):
    mel = librosa.feature.melspectrogram(
        y=y, sr=TARGET_SR, n_mels=N_MELS, n_fft=N_FFT,
        hop_length=HOP_LENGTH, win_length=WIN_LENGTH, power=2.0
    )
    logmel = librosa.power_to_db(mel, ref=np.max)
    return logmel.astype(np.float32)

fv = extract_feature_vector(model_ex)
logmel = extract_logmel_image(model_ex)
print("Feature vector dim:", fv.shape)
print("Log-Mel shape:", logmel.shape)

In [ ]:
examples = metadata.groupby("emotion", group_keys=False).sample(1, random_state=SEED)
for row in examples.itertuples(index=False):
    raw, clean, model, _ = preprocess_versions(row.filepath)
    stft_db = librosa.amplitude_to_db(np.abs(librosa.stft(model, n_fft=1024, hop_length=256)), ref=np.max)
    mfcc = librosa.feature.mfcc(y=model, sr=TARGET_SR, n_mfcc=40, n_fft=1024, hop_length=256)
    mel_db = extract_logmel_image(model)

    fig, axes = plt.subplots(4, 1, figsize=(13, 12))
    librosa.display.waveshow(model, sr=TARGET_SR, ax=axes[0])
    axes[0].set_title(f"Waveform model-ready 4s - {row.emotion} ({row.dataset})")
    librosa.display.specshow(stft_db, sr=TARGET_SR, hop_length=256, x_axis="time", y_axis="hz", ax=axes[1])
    axes[1].set_title("Spectrogram")
    librosa.display.specshow(mfcc, sr=TARGET_SR, hop_length=256, x_axis="time", ax=axes[2])
    axes[2].set_title("MFCC")
    librosa.display.specshow(mel_db, sr=TARGET_SR, hop_length=HOP_LENGTH, x_axis="time", y_axis="mel", ax=axes[3])
    axes[3].set_title("Log-Mel spectrogram")
    plt.tight_layout()
    fig_path = FIGURE_DIR / f"feature_example_{row.emotion}_{row.dataset}.png"
    plt.savefig(fig_path, dpi=150)
    plt.show()
    print("Saved:", fig_path)

## 8. Batch Export Artifacts

Kết quả chính:

- `metadata.csv`: metadata đầy đủ, split, label, speaker/domain id, audio statistics;
- `features/baseline_features_v2.npz`: vector feature cho baseline classical ML;
- `audio_16k_raw/`: raw 16 kHz;
- `audio_16k_clean_full/`: clean full length;
- `audio_16k/`: model-ready 4s, tương thích notebook 04/05/06;
- `processing_config.json`: cấu hình xử lý;
- `reports/`: overview, validation, split, feature description;
- `figures/`: hình minh họa.

Nếu muốn tiết kiệm dung lượng Kaggle, có thể đặt:

```text
CACHE_AUDIO=0
```

khi đó notebook vẫn tạo metadata/features nhưng không ghi WAV cache.

In [ ]:
feature_vectors, labels, sample_ids, splits = [], [], [], []
audio_stat_rows = []
failed = []

for row in tqdm(metadata.itertuples(index=False), total=len(metadata), desc="Export audio/features"):
    try:
        raw, clean, model, stats = preprocess_versions(row.filepath)
        fv = extract_feature_vector(model)

        feature_vectors.append(fv)
        labels.append(row.emotion)
        sample_ids.append(row.sample_id)
        splits.append(row.split)
        audio_stat_rows.append({"sample_id": row.sample_id, **stats})

        if CACHE_AUDIO:
            sf.write(RAW_AUDIO_DIR / f"{row.sample_id}.wav", raw, TARGET_SR)
            sf.write(CLEAN_AUDIO_DIR / f"{row.sample_id}.wav", clean, TARGET_SR)
            sf.write(MODEL_AUDIO_DIR / f"{row.sample_id}.wav", model, TARGET_SR)
    except Exception as exc:
        failed.append({"sample_id": row.sample_id, "filepath": row.filepath, "error": str(exc)})

if failed:
    failed_df = pd.DataFrame(failed)
    failed_df.to_csv(REPORT_DIR / "failed_processing.csv", index=False)
    print("Failed files:", len(failed_df))

audio_stats_df = pd.DataFrame(audio_stat_rows)
metadata = metadata.merge(audio_stats_df, on="sample_id", how="left")

X = np.vstack(feature_vectors).astype(np.float32)
y_labels = np.asarray(labels)
sample_ids_arr = np.asarray(sample_ids)
splits_arr = np.asarray(splits)

scaler = StandardScaler()
train_mask = splits_arr == "train"
X_scaled = np.empty_like(X, dtype=np.float32)
if train_mask.any():
    X_scaled[train_mask] = scaler.fit_transform(X[train_mask]).astype(np.float32)
    X_scaled[~train_mask] = scaler.transform(X[~train_mask]).astype(np.float32)

np.savez_compressed(
    FEATURE_DIR / "baseline_features_v2.npz",
    X=X,
    X_scaled=X_scaled,
    y=y_labels,
    sample_id=sample_ids_arr,
    split=splits_arr,
    feature_dim=np.asarray([X.shape[1]], dtype=np.int32),
    scaler_mean=scaler.mean_ if hasattr(scaler, "mean_") else np.array([]),
    scaler_scale=scaler.scale_ if hasattr(scaler, "scale_") else np.array([]),
)

metadata.to_csv(OUTPUT_DIR / "metadata.csv", index=False)

feature_description = pd.DataFrame([
    ["MFCC + delta + delta-delta", "baseline vector; 06D/06F temporal branch", "spectral envelope and temporal dynamics", "captures tone, articulation and vocal color changes"],
    ["RMS / energy", "baseline vector; 06D/06F temporal/stats", "frame-level loudness", "emotion often changes intensity"],
    ["ZCR", "baseline vector; 06D/06F temporal/stats", "sign-crossing rate", "captures voiced/unvoiced texture and harshness"],
    ["Spectral centroid/bandwidth/rolloff/contrast/flatness", "baseline vector; 06D/06F temporal/stats", "brightness, spread, high-frequency concentration, peak-valley contrast, noisiness", "captures tense, sharp, soft or noisy speech texture"],
    ["Chroma STFT", "baseline vector; 06D_update/06F advanced stats", "harmonic pitch-class distribution; CQT/CENS are optional future variants, not required in the current notebook", "adds tonal/harmonic information beyond MFCC"],
    ["Pitch/F0 + voiced ratio", "baseline vector; 06D_update/06F advanced stats and feedback indicators", "fundamental frequency and voiced activity", "emotion strongly affects pitch range and pitch variability"],
    ["Pause/speech ratio", "metadata/feedback indicator", "amount of silence or active speech", "useful for nervousness, fluency and presentation feedback"],
    ["Jitter/shimmer approximations", "baseline vector; advanced stats", "micro-variation of pitch/amplitude", "approximates voice quality/tension"],
    ["Statistical pooling", "stats branch and baseline vector", "mean, std, min, max, median, IQR and slope summaries over time", "turns variable-length frame features into one stable vector for SVM/RF/MLP-style models"],
    ["Log-Mel + delta + delta-delta Log-Mel", "06D/06F spectrogram branch", "2D time-frequency energy map plus first/second temporal change maps", "lets CNN or pretrained spectral encoders learn local spectral-temporal patterns"],
    ["Raw waveform", "emotion2vec branch", "original time-domain speech signal", "pretrained emotion2vec extracts high-level emotion representation"],
    ["LPC / formant-style features", "not used in the current 06D implementation; future optional stats feature", "linear prediction coefficients approximate vocal-tract resonance", "can add complementary voice-production information, but needs careful validation"],
], columns=["Feature group", "Used by branch", "Meaning", "Why it matters"])
feature_description.to_csv(REPORT_DIR / "speech_feature_description_v2.csv", index=False)

config = {
    "notebook": "0102_update_data_processing",
    "kaggle_dataset_slug": KAGGLE_DATASET_SLUG,
    "data_source_mode": DATA_SOURCE_MODE,
    "data_root": str(DATA_ROOT),
    "target_sr": TARGET_SR,
    "target_duration": TARGET_DURATION,
    "common_emotions": COMMON_EMOTIONS,
    "audio_versions": {
        "audio_16k_raw": "mono 16k, DC removed, no trim/normalization",
        "audio_16k_clean_full": "trim light silence + RMS normalization + peak guard, natural length",
        "audio_16k": "model-ready 4s crop/pad from clean_full",
    },
    "split_strategy": "GroupShuffleSplit by speaker_global",
    "random_state": SEED,
    "feature_dim": int(X.shape[1]),
    "cache_audio": CACHE_AUDIO,
    "dataset_counts": metadata["dataset"].value_counts().to_dict(),
    "emotion_counts": metadata["emotion"].value_counts().to_dict(),
}
with open(OUTPUT_DIR / "processing_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

display(feature_description)
print("metadata:", metadata.shape)
print("X:", X.shape)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
print("Final metadata shape:", metadata.shape)
print("Duplicate filepath:", metadata["filepath"].duplicated().sum())
print("Duplicate dataset+source_filename:", metadata.duplicated(["dataset", "source_filename"]).sum())
print("Missing emotion:", metadata["emotion"].isna().sum())
print("Datasets:", metadata["dataset"].value_counts().to_dict())
print("Splits:", metadata["split"].value_counts().to_dict())
print("Audio dirs exist:", RAW_AUDIO_DIR.exists(), CLEAN_AUDIO_DIR.exists(), MODEL_AUDIO_DIR.exists())
print("Artifacts:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(" -", p.name)

## 9. Package Output

Mặc định cell này tạo **light zip** không chứa audio WAV để tránh file quá nặng:

```text
0102_update_ser_processed_light_no_audio.zip
```

Zip nhẹ gồm:

- `metadata.csv`
- `processing_config.json`
- `features/`
- `reports/`
- `figures/`

Nếu thật sự muốn zip cả audio, đặt:

```text
PACKAGE_FULL_AUDIO=1
```

nhưng file có thể rất lớn.

In [ ]:
light_zip_path = PROJECT_ROOT / "0102_update_ser_processed_light_no_audio.zip"
include_items = [
    OUTPUT_DIR / "metadata.csv",
    OUTPUT_DIR / "processing_config.json",
    FEATURE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
]

with zipfile.ZipFile(light_zip_path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
    for item in include_items:
        if not item.exists():
            print("Skip missing:", item)
            continue
        if item.is_file():
            zf.write(item, item.relative_to(OUTPUT_DIR.parent))
        else:
            count = 0
            for file_path in item.rglob("*"):
                if file_path.is_file():
                    zf.write(file_path, file_path.relative_to(OUTPUT_DIR.parent))
                    count += 1
            print("Added", item.name, count, "files")

with zipfile.ZipFile(light_zip_path, "r") as zf:
    bad_file = zf.testzip()
    names = zf.namelist()

print("LIGHT ZIP:", light_zip_path)
print("ZIP size MB:", round(light_zip_path.stat().st_size / (1024 * 1024), 2))
print("ZIP integrity:", "OK" if bad_file is None else f"BAD FILE: {bad_file}")
print("First files:", names[:20])

if os.getenv("PACKAGE_FULL_AUDIO", "0") == "1":
    full_zip_path = PROJECT_ROOT / "0102_update_ser_processed_FULL_WITH_AUDIO.zip"
    with zipfile.ZipFile(full_zip_path, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        for file_path in OUTPUT_DIR.rglob("*"):
            if file_path.is_file():
                zf.write(file_path, file_path.relative_to(OUTPUT_DIR.parent))
    print("FULL ZIP:", full_zip_path)
    print("FULL ZIP size MB:", round(full_zip_path.stat().st_size / (1024 * 1024), 2))

light_zip_path